In [25]:
###### INSTALLING LIBRARIES 
pip install pandas geopandas rasterio numpy matplotlib scikit-learn pygbif shapely

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# CORRELATION ANALYSIS FOR Trachypithecus phayrei
# SPECIE_01_PD0602_Trachypithecus_phayrei
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import matplotlib.pyplot as plt

SPECIES = "Trachypithecus phayrei"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_01_PD0602_Trachypithecus_phayrei"

OUT_DIR = PROJECT_DIR / "02_results" / "correlation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Trachypithecus phayrei.csv"
CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

COMMON_BIO_IDS = [1, 4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
CORR_THRESHOLD = 0.75

def ensure_exists(path, label):
    if not Path(path).exists():
        raise FileNotFoundError(f"{label} not found: {path}")

def read_occurrences(csv_path):
    df = pd.read_csv(csv_path)

    lon_candidates = ["decimalLongitude", "longitude", "lon", "x"]
    lat_candidates = ["decimalLatitude", "latitude", "lat", "y"]

    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Available columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].copy()
    df.columns = ["lon", "lat"]
    df = df.dropna()

    df = df[
        (df["lon"] >= -180) & (df["lon"] <= 180) &
        (df["lat"] >= -90) & (df["lat"] <= 90)
    ].copy()

    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

def remove_duplicate_coords(gdf):
    gdf = gdf.copy()
    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)
    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    return gdf.drop(columns=["x_round", "y_round"])

def clip_points_to_study_area(gdf, bounds):
    minx, miny, maxx, maxy = bounds
    return gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

def build_study_polygon(bounds):
    minx, miny, maxx, maxy = bounds
    return gpd.GeoDataFrame(
        {"name": ["study_region"]},
        geometry=[box(minx, miny, maxx, maxy)],
        crs="EPSG:4326"
    )

def find_bio_layers(folder, bio_ids):
    tif_files = list(Path(folder).glob("*.tif"))
    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []
        for p in tif_files:
            stem = p.stem.lower().replace("-", "_")
            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(p)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO layer {bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def crop_and_mask_single_raster(src_path, shapes_gdf, out_path):
    with rasterio.open(src_path) as src:
        out_img, out_transform = mask(src, list(shapes_gdf.geometry), crop=True)
        out_meta = src.meta.copy()

        out_meta.update({
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_img)

    return out_path

def sample_raster_values(raster_paths, points_gdf):
    data = pd.DataFrame(index=points_gdf.index)
    coords = [(geom.x, geom.y) for geom in points_gdf.geometry]

    for bio_id, path in raster_paths.items():
        with rasterio.open(path) as src:
            vals = np.array([v[0] for v in src.sample(coords)], dtype=float)

            nodata = src.nodata
            if nodata is not None:
                vals = np.where(vals == nodata, np.nan, vals)

            vals = np.where(np.isinf(vals), np.nan, vals)
            data[f"bio_{bio_id}"] = vals

    return data

def find_high_correlation_pairs(corr_matrix, threshold):
    rows = []
    cols = list(corr_matrix.columns)

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            var1 = cols[i]
            var2 = cols[j]
            corr_val = corr_matrix.loc[var1, var2]

            if pd.notna(corr_val) and abs(corr_val) > threshold:
                rows.append({
                    "Variable_1": var1,
                    "Variable_2": var2,
                    "Correlation": corr_val,
                    "Abs_Correlation": abs(corr_val)
                })

    if rows:
        return pd.DataFrame(rows).sort_values("Abs_Correlation", ascending=False)

    return pd.DataFrame(columns=["Variable_1", "Variable_2", "Correlation", "Abs_Correlation"])

def remove_correlated_variables(df, threshold=0.75):
    corr = df.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = []
    for col in upper.columns:
        if any(upper[col] > threshold):
            to_drop.append(col)

    keep = [c for c in df.columns if c not in to_drop]
    return keep

def main():
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(CURRENT_DIR, "Current climate folder")

    print("SPECIES:", SPECIES)
    print("GBIF_CSV:", GBIF_CSV)
    print("CURRENT_DIR:", CURRENT_DIR)
    print("OUT_DIR:", OUT_DIR)

    occ = read_occurrences(GBIF_CSV)
    print("Raw occurrence points:", len(occ))

    occ = clip_points_to_study_area(occ, STUDY_BOUNDS)
    print("Occurrence points inside study area:", len(occ))

    occ_unique = remove_duplicate_coords(occ)
    print("Points after duplicate removal:", len(occ_unique))

    if len(occ_unique) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    study_region = build_study_polygon(STUDY_BOUNDS)

    current_layers_raw = find_bio_layers(CURRENT_DIR, COMMON_BIO_IDS)

    print("\nCandidate BIO layers:")
    for k, v in current_layers_raw.items():
        print(f"BIO{k}: {v}")

    crop_dir = OUT_DIR / "current_crop_for_correlation"
    crop_dir.mkdir(parents=True, exist_ok=True)

    current_layers = {}
    for bio_id, p in current_layers_raw.items():
        current_layers[bio_id] = crop_and_mask_single_raster(
            p,
            study_region,
            crop_dir / f"bio_{bio_id}.tif"
        )

    env_at_points = sample_raster_values(current_layers, occ_unique)
    complete = env_at_points.dropna().copy()

    print("Occurrence points with complete raster values:", len(complete))

    if complete.empty:
        raise ValueError("No valid raster values extracted at occurrence points.")

    corr_matrix = complete.corr(numeric_only=True)
    high_corr_pairs = find_high_correlation_pairs(corr_matrix, CORR_THRESHOLD)
    retained_vars = remove_correlated_variables(complete, threshold=CORR_THRESHOLD)

    all_vars = list(corr_matrix.columns)
    dropped_vars = [v for v in all_vars if v not in retained_vars]

    summary_df = pd.DataFrame({
        "Variable": all_vars,
        "Status": ["Retained" if v in retained_vars else "Dropped" for v in all_vars]
    })

    selected_df = pd.DataFrame({
        "kept_variable": retained_vars,
        "bio_id": [int(v.split("_")[1]) for v in retained_vars]
    })

    corr_xlsx = OUT_DIR / f"{SPECIES_SAFE}_correlation_matrix.xlsx"
    high_pairs_xlsx = OUT_DIR / f"{SPECIES_SAFE}_high_correlation_pairs.xlsx"
    summary_xlsx = OUT_DIR / f"{SPECIES_SAFE}_variable_selection_summary.xlsx"
    selected_csv = OUT_DIR / f"{SPECIES_SAFE}_selected_variables.csv"

    corr_matrix.to_excel(corr_xlsx)
    high_corr_pairs.to_excel(high_pairs_xlsx, index=False)
    summary_df.to_excel(summary_xlsx, index=False)
    selected_df.to_csv(selected_csv, index=False)

    heatmap_png = OUT_DIR / f"{SPECIES_SAFE}_correlation_heatmap.png"

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr_matrix.values, interpolation="nearest")

    ax.set_xticks(range(len(corr_matrix.columns)))
    ax.set_yticks(range(len(corr_matrix.index)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr_matrix.index)

    for i in range(corr_matrix.shape[0]):
        for j in range(corr_matrix.shape[1]):
            ax.text(
                j,
                i,
                f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Pearson correlation")

    ax.set_title(f"{SPECIES} - Correlation Heatmap", fontsize=14)
    plt.tight_layout()
    plt.savefig(heatmap_png, dpi=300, bbox_inches="tight")
    plt.close()

    print("\nDONE ✅ CORRELATION ANALYSIS")
    print("Retained variables:", retained_vars)
    print("Dropped variables:", dropped_vars)
    print("Selected variables CSV:", selected_csv)
    print("Heatmap:", heatmap_png)

if __name__ == "__main__":
    main()

SPECIES: Trachypithecus phayrei
GBIF_CSV: C:\Users\ual-laptop\Desktop\CAPSTONE\data\gbif\Trachypithecus phayrei.csv
CURRENT_DIR: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min
OUT_DIR: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\correlation
Raw occurrence points: 57
Occurrence points inside study area: 57
Points after duplicate removal: 56

Candidate BIO layers:
BIO1: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_1.tif
BIO4: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_4.tif
BIO8: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_8.tif
BIO9: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_9.tif
BIO10: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_10.tif
BIO11: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_11.tif
BIO12: C:\Users\ual-laptop\Desktop\CAPSTONE\Time Period\Current\10min\bio_12.tif
BIO13: C:\Users\

In [4]:
# ============================================================
# FULL PAST SDM PIPELINE FOR Trachypithecus phayrei
# SPECIE_01_PD0602
# Uses correlation-selected variables automatically
# Produces 4-panel past SDM map with auto-stretch display
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from rasterio.features import geometry_mask
from shapely.geometry import box
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Trachypithecus phayrei"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_01_PD0602_Trachypithecus_phayrei"

OUT_DIR = PROJECT_DIR / "02_results" / "past"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CORR_DIR = PROJECT_DIR / "02_results" / "correlation"
SELECTED_CSV = CORR_DIR / f"{SPECIES_SAFE}_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Trachypithecus phayrei.csv"
COUNTRIES_SHP = BASE_DIR / "data" / "boundaries" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"
LGM_DIR = BASE_DIR / "Time Period" / "Last Glacial Maximum" / "10min"
PLEISTOCENE_DIR = BASE_DIR / "Time Period" / "Pleistocene" / "10min"
PLIOCENE_DIR = BASE_DIR / "Time Period" / "Pliocene" / "10min"

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

MAXENT_REPLICATES = 10
MAXENT_RANDOM_TEST = 25
MAXENT_BACKGROUND = 10000
MAXENT_ITERATIONS = 5000
MAXENT_OUTPUT_FORMAT = "cloglog"
REG_MULTIPLIER = 1.0

ASCII_NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"

# ============================================================
# BASIC HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def get_safe_nodata(src_nodata):
    if src_nodata is None:
        return ASCII_NODATA
    try:
        val = float(src_nodata)
        if np.isfinite(val):
            return val
    except Exception:
        pass
    return ASCII_NODATA

# ============================================================
# LOAD SELECTED VARIABLES FROM CORRELATION RESULT
# ============================================================

def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Correlation selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)

    if "selected_variable" in df.columns:
        col = "selected_variable"
    elif "kept_variable" in df.columns:
        col = "kept_variable"
    elif "Variable" in df.columns and "Status" in df.columns:
        df = df[df["Status"].astype(str).str.lower() == "retained"]
        col = "Variable"
    else:
        col = df.columns[0]

    selected = df[col].dropna().astype(str).tolist()

    bio_ids = []
    for v in selected:
        v = v.strip().lower().replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO variables found in selected variables file.")

    print("Using correlation-selected BIO layers:", bio_ids)
    return bio_ids

# ============================================================
# OCCURRENCE DATA
# ============================================================

def read_occurrences(csv_path):
    df = pd.read_csv(csv_path)

    lon_col = next((c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Available columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    df = df[
        (df["lon"] >= -180) & (df["lon"] <= 180) &
        (df["lat"] >= -90) & (df["lat"] <= 90)
    ].copy()

    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

def remove_duplicate_coords(points_gdf):
    points_gdf = points_gdf.copy()
    points_gdf["x_round"] = points_gdf.geometry.x.round(6)
    points_gdf["y_round"] = points_gdf.geometry.y.round(6)
    points_gdf = points_gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    return points_gdf.drop(columns=["x_round", "y_round"])

def clip_points_to_study_area(points_gdf, bounds):
    minx, miny, maxx, maxy = bounds
    return points_gdf[
        (points_gdf.geometry.x >= minx) &
        (points_gdf.geometry.x <= maxx) &
        (points_gdf.geometry.y >= miny) &
        (points_gdf.geometry.y <= maxy)
    ].copy()

def export_samples(points_gdf, out_csv):
    df = pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    })
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    return out_csv

# ============================================================
# SPATIAL HELPERS
# ============================================================

def build_study_polygon(bounds):
    minx, miny, maxx, maxy = bounds
    return gpd.GeoDataFrame(
        {"name": ["study_region"]},
        geometry=[box(minx, miny, maxx, maxy)],
        crs="EPSG:4326"
    )

def load_countries(path):
    countries = gpd.read_file(path)
    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")
    return countries.to_crs("EPSG:4326")

# ============================================================
# BIO LAYER HANDLING
# ============================================================

def find_bio_layers(folder, bio_ids):
    folder = Path(folder)
    tif_files = list(folder.glob("*.tif"))

    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for p in tif_files:
            stem = p.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(p)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def crop_single_raster(src_path, study_region, out_path):
    with rasterio.open(src_path) as src:
        out_img, out_transform = mask(
            src,
            list(study_region.geometry),
            crop=True,
            filled=False
        )

        nodata = get_safe_nodata(src.nodata)
        band = np.ma.array(out_img[0], copy=False)
        arr = band.filled(nodata).astype(np.float32)
        arr = np.where(np.isfinite(arr), arr, nodata).astype(np.float32)

        meta = src.meta.copy()
        meta.update({
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": out_transform,
            "dtype": "float32",
            "nodata": nodata
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

    return out_path

def save_ascii_from_tif(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True)

        nodata = get_safe_nodata(src.nodata)
        arr = np.ma.array(band, copy=False).filled(nodata).astype(np.float32)
        arr = np.where(np.isfinite(arr), arr, nodata).astype(np.float32)

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {nodata}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")

    return out_asc

def validate_ascii_file(path):
    with open(path, "r") as f:
        lines = f.readlines()

    if len(lines) < 7:
        raise ValueError(f"ASCII file too short: {path}")

    header = lines[:6]
    data = lines[6:]

    ncols = int(header[0].split()[1])
    nrows = int(header[1].split()[1])

    if len(data) != nrows:
        raise ValueError(f"{path.name}: expected {nrows} data rows, found {len(data)}")

    for i, row in enumerate(data, start=1):
        values = row.strip().split()

        if len(values) != ncols:
            raise ValueError(f"{path.name}: row {i} expected {ncols} columns, found {len(values)}")

        if any(v.lower() == "nan" for v in values):
            raise ValueError(f"{path.name}: contains literal nan on row {i}")

def validate_ascii_directory(ascii_dir):
    ascii_dir = Path(ascii_dir)
    asc_files = list(ascii_dir.glob("*.asc"))

    if not asc_files:
        raise ValueError(f"No ASCII files found in {ascii_dir}")

    for asc in asc_files:
        validate_ascii_file(asc)

# ============================================================
# MAXENT
# ============================================================

def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(samples_csv, "samples CSV")
    ensure_exists(env_dir, "environment directory")
    ensure_exists(projection_dir, "projection directory")

    validate_ascii_directory(env_dir)
    validate_ascii_directory(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        f"randomtestpoints={MAXENT_RANDOM_TEST}",
        f"maximumbackground={MAXENT_BACKGROUND}",
        f"replicates={MAXENT_REPLICATES}",
        "replicatetype=subsample",
        f"maximumiterations={MAXENT_ITERATIONS}",
        f"outputformat={MAXENT_OUTPUT_FORMAT}",
        f"betamultiplier={REG_MULTIPLIER}",
        "autofeature=true",
        "responsecurves=true",
        "jackknife=true",
        "pictures=true",
        "writebackgroundpredictions=false",
        "visible=false",
        "autorun",
        "redoifexists",
        "novisible"
    ]

    print("\n============================================================")
    print("Running MaxEnt")
    print("============================================================")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- MaxEnt STDOUT ---")
    print(result.stdout)

    print("\n--- MaxEnt STDERR ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check STDERR output above.")

def find_prediction(output_dir):
    output_dir = Path(output_dir)

    tif_candidates = list(output_dir.glob("*.tif"))
    asc_candidates = list(output_dir.glob("*.asc"))

    candidates = tif_candidates + asc_candidates

    candidates = [
        p for p in candidates
        if "clamping" not in p.name.lower()
        and "novel" not in p.name.lower()
        and "limiting" not in p.name.lower()
        and "background" not in p.name.lower()
    ]

    if not candidates:
        raise FileNotFoundError(f"No prediction raster found in {output_dir}")

    ranked = sorted(
        candidates,
        key=lambda p: (
            "median" not in p.stem.lower(),
            p.name.lower()
        )
    )

    chosen = ranked[0]
    print("Prediction chosen:", chosen)

    return chosen

# ============================================================
# PLOTTING
# ============================================================

def load_land_masked_prediction(raster_path, countries):
    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan

        extent = plotting_extent(src)

        xmin, xmax, ymin, ymax = extent
        raster_box = box(xmin, ymin, xmax, ymax)

        countries_here = countries[countries.intersects(raster_box)].copy()

        if not countries_here.empty:
            raster_gdf = gpd.GeoDataFrame(
                {"name": ["raster_extent"]},
                geometry=[raster_box],
                crs="EPSG:4326"
            )

            countries_here = gpd.clip(countries_here, raster_gdf)

            land_mask = geometry_mask(
                countries_here.geometry,
                transform=src.transform,
                invert=True,
                out_shape=arr.shape
            )

            arr[~land_mask] = np.nan

    return arr, extent, countries_here

def plot_auto_stretch_sdm(raster_path, countries, title, out_png):
    arr, extent, boundaries = load_land_masked_prediction(raster_path, countries)

    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        raise ValueError(f"No valid raster values found in {raster_path}")

    vmin = np.percentile(finite, 2)
    vmax = np.percentile(finite, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    minx, miny, maxx, maxy = STUDY_BOUNDS

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Habitat suitability (auto-stretched)")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

def make_four_panel(current_png, lgm_png, pleistocene_png, pliocene_png, out_png):
    imgs = [
        ("Current", plt.imread(current_png)),
        ("Last Glacial Maximum", plt.imread(lgm_png)),
        ("Pleistocene", plt.imread(pleistocene_png)),
        ("Pliocene", plt.imread(pliocene_png)),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()

    for ax, (label, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(label, fontsize=12)

    fig.suptitle(SPECIES, fontsize=18)
    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    ensure_exists(SELECTED_CSV, "Correlation selected variables file")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LGM_DIR, "Last Glacial Maximum folder")
    ensure_exists(PLEISTOCENE_DIR, "Pleistocene folder")
    ensure_exists(PLIOCENE_DIR, "Pliocene folder")

    print("Output folder:", OUT_DIR)

    selected_bio_ids = load_selected_bio_ids()

    occ = read_occurrences(GBIF_CSV)
    print("Raw occurrence points:", len(occ))

    occ = clip_points_to_study_area(occ, STUDY_BOUNDS)
    print("Occurrence points inside study area:", len(occ))

    occ = remove_duplicate_coords(occ)
    print("Occurrence points after duplicate removal:", len(occ))

    if len(occ) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(
        occ,
        OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"
    )

    study_region = build_study_polygon(STUDY_BOUNDS)
    countries = load_countries(COUNTRIES_SHP)

    print("\nFinding BIO layers...")

    current_layers = find_bio_layers(CURRENT_DIR, selected_bio_ids)
    lgm_layers = find_bio_layers(LGM_DIR, selected_bio_ids)
    pleistocene_layers = find_bio_layers(PLEISTOCENE_DIR, selected_bio_ids)
    pliocene_layers = find_bio_layers(PLIOCENE_DIR, selected_bio_ids)

    crop_dirs = {
        "current": OUT_DIR / "current_crop",
        "lgm": OUT_DIR / "lgm_crop",
        "pleistocene": OUT_DIR / "pleistocene_crop",
        "pliocene": OUT_DIR / "pliocene_crop",
    }

    ascii_dirs = {
        "current": OUT_DIR / "ascii_current",
        "lgm": OUT_DIR / "ascii_lgm",
        "pleistocene": OUT_DIR / "ascii_pleistocene",
        "pliocene": OUT_DIR / "ascii_pliocene",
    }

    for folder in list(crop_dirs.values()) + list(ascii_dirs.values()):
        remove_and_recreate_dir(folder)

    print("\nCropping rasters...")

    for bio_id, src in current_layers.items():
        crop_single_raster(src, study_region, crop_dirs["current"] / f"bio_{bio_id}.tif")

    for bio_id, src in lgm_layers.items():
        crop_single_raster(src, study_region, crop_dirs["lgm"] / f"bio_{bio_id}.tif")

    for bio_id, src in pleistocene_layers.items():
        crop_single_raster(src, study_region, crop_dirs["pleistocene"] / f"bio_{bio_id}.tif")

    for bio_id, src in pliocene_layers.items():
        crop_single_raster(src, study_region, crop_dirs["pliocene"] / f"bio_{bio_id}.tif")

    print("\nConverting cropped rasters to ASCII...")

    for key in crop_dirs:
        for tif in crop_dirs[key].glob("*.tif"):
            save_ascii_from_tif(
                tif,
                ascii_dirs[key] / tif.with_suffix(".asc").name
            )

    print("\nASCII file counts:")
    for key, folder in ascii_dirs.items():
        print(key, len(list(folder.glob("*.asc"))))

    maxent_current = OUT_DIR / "maxent_current"
    maxent_lgm = OUT_DIR / "maxent_lgm"
    maxent_pleistocene = OUT_DIR / "maxent_pleistocene"
    maxent_pliocene = OUT_DIR / "maxent_pliocene"

    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["current"], maxent_current)
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["lgm"], maxent_lgm)
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["pleistocene"], maxent_pleistocene)
    run_maxent(samples_csv, ascii_dirs["current"], ascii_dirs["pliocene"], maxent_pliocene)

    print("\nFinding MaxEnt prediction rasters...")

    current_pred = find_prediction(maxent_current)
    lgm_pred = find_prediction(maxent_lgm)
    pleistocene_pred = find_prediction(maxent_pleistocene)
    pliocene_pred = find_prediction(maxent_pliocene)

    current_png = OUT_DIR / f"{SPECIES_SAFE}_current_auto_stretch.png"
    lgm_png = OUT_DIR / f"{SPECIES_SAFE}_lgm_auto_stretch.png"
    pleistocene_png = OUT_DIR / f"{SPECIES_SAFE}_pleistocene_auto_stretch.png"
    pliocene_png = OUT_DIR / f"{SPECIES_SAFE}_pliocene_auto_stretch.png"

    combined_png = OUT_DIR / f"{SPECIES_SAFE}_4panel_past_auto_stretch.png"

    print("\nCreating maps...")

    plot_auto_stretch_sdm(
        current_pred,
        countries,
        f"{SPECIES} - Current SDM",
        current_png
    )

    plot_auto_stretch_sdm(
        lgm_pred,
        countries,
        f"{SPECIES} - Last Glacial Maximum SDM",
        lgm_png
    )

    plot_auto_stretch_sdm(
        pleistocene_pred,
        countries,
        f"{SPECIES} - Pleistocene SDM",
        pleistocene_png
    )

    plot_auto_stretch_sdm(
        pliocene_pred,
        countries,
        f"{SPECIES} - Pliocene SDM",
        pliocene_png
    )

    make_four_panel(
        current_png,
        lgm_png,
        pleistocene_png,
        pliocene_png,
        combined_png
    )

    print("\n============================================================")
    print("DONE ✅")
    print("============================================================")
    print("Final 4-panel past map saved here:")
    print(combined_png)

if __name__ == "__main__":
    main()

Output folder: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\past
Using correlation-selected BIO layers: [1, 4, 8, 10, 12, 15]
Raw occurrence points: 57
Occurrence points inside study area: 57
Occurrence points after duplicate removal: 56

Finding BIO layers...

Cropping rasters...

Converting cropped rasters to ASCII...

ASCII file counts:
current 6
lgm 6
pleistocene 6
pliocene 6

Running MaxEnt
C:\Program Files\Java\jdk-17\bin\java.exe -Xmx4g -jar C:\Users\ual-laptop\Desktop\CAPSTONE\maxent\maxent.jar samplesfile=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\past\Trachypithecus_phayrei_maxent_samples.csv environmentallayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\past\ascii_current projectionlayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\past\ascii_current outputdirectory=C:\Users\ual-laptop\Desktop\CAPSTON

In [2]:
# ============================================================
# FULL FUTURE SDM PIPELINE FOR Trachypithecus phayrei
# SPECIE_01_PD0602
#
# Includes:
# 1) Uses future-correlation selected BIO layers
# 2) Crops current + SSP126 + SSP585 layers
# 3) Converts to ASCII
# 4) Runs MaxEnt
# 5) Creates ONLY 2-panel future map
# 6) Removes ugly 1e-5 scale using relative 0–1 display
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from shapely.geometry import box
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Trachypithecus phayrei"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_01_PD0602_Trachypithecus_phayrei"

OUT_DIR = PROJECT_DIR / "02_results" / "future"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FUTURE_CORR_DIR = PROJECT_DIR / "02_results" / "future_correlation"
SELECTED_CSV = FUTURE_CORR_DIR / f"{SPECIES_SAFE}_future_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Trachypithecus phayrei.csv"
COUNTRIES_SHP = BASE_DIR / "data" / "boundaries" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

LOW_FUTURE_TIF = BASE_DIR / "Time Period" / "Future_CMIP6" / "MIROC6" / "wc2.1_2.5m_bioc_MIROC6_ssp126_2081-2100.tif"
HIGH_FUTURE_TIF = BASE_DIR / "Time Period" / "Future_CMIP6" / "MIROC6" / "wc2.1_2.5m_bioc_MIROC6_ssp585_2081-2100.tif"

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"

# ============================================================
# OUTPUT FILES
# ============================================================

SSP126_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP126_2panel_clear.png"
SSP585_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP585_2panel_clear.png"
FINAL_2PANEL = OUT_DIR / f"{SPECIES_SAFE}_Future_2panel_CLEAR_no_scientific_scale.png"

# ============================================================
# BASIC HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Future selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)

    if "selected_variable" in df.columns:
        col = "selected_variable"
    else:
        col = df.columns[0]

    bio_ids = []

    for v in df[col].dropna():
        v = str(v).strip().lower().replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO variables found.")

    print("Using future selected BIO layers:", bio_ids)
    return bio_ids

def read_occurrences():
    df = pd.read_csv(GBIF_CSV)

    lon_col = next((c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns), None)
    lat_col = next((c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    minx, miny, maxx, maxy = STUDY_BOUNDS

    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)

    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    gdf = gdf.drop(columns=["x_round", "y_round"])

    print("Clean occurrence points:", len(gdf))
    return gdf

def export_samples(points_gdf):
    samples_csv = OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"

    pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    }).to_csv(samples_csv, index=False)

    return samples_csv

def build_study_polygon():
    return gpd.GeoDataFrame(
        geometry=[box(*STUDY_BOUNDS)],
        crs="EPSG:4326"
    )

def load_countries():
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")

    countries = gpd.read_file(COUNTRIES_SHP)

    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")

    return countries.to_crs("EPSG:4326")

# ============================================================
# RASTER PREPARATION
# ============================================================

def find_current_bio_layers(folder, bio_ids):
    folder = Path(folder)
    tif_files = list(folder.glob("*.tif"))

    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for p in tif_files:
            stem = p.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(p)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping

def crop_current_raster(src_path, out_path, region):
    with rasterio.open(src_path) as src:
        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            filled=False
        )

        arr = np.ma.array(img[0], copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

def extract_future_band(src_tif, bio_id, region, out_tif):
    with rasterio.open(src_tif) as src:
        if bio_id < 1 or bio_id > src.count:
            raise ValueError(f"BIO{bio_id} requested, but future raster has only {src.count} bands.")

        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            indexes=bio_id,
            filled=False
        )

        arr = np.ma.array(img, copy=False).astype("float32")
        arr = arr.filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": NODATA
        })

        out_tif.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(arr, 1)

def tif_to_ascii(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True).astype("float32")

        arr = np.ma.array(band, copy=False).filled(NODATA).astype("float32")
        arr = np.where(np.isfinite(arr), arr, NODATA).astype("float32")

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {NODATA}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")

def validate_ascii_dir(folder):
    asc_files = list(Path(folder).glob("*.asc"))

    if not asc_files:
        raise ValueError(f"No ASCII files found in {folder}")

# ============================================================
# MAXENT
# ============================================================

def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    validate_ascii_dir(env_dir)
    validate_ascii_dir(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        "outputformat=cloglog",
        "randomtestpoints=25",
        "maximumbackground=10000",
        "maximumiterations=5000",
        "autorun",
        "redoifexists",
        "novisible",
        "pictures=false",
        "responsecurves=false",
        "jackknife=false",
        "writebackgroundpredictions=false"
    ]

    print("\nRunning MaxEnt:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check output above.")

# ============================================================
# PLOTTING
# ============================================================

def find_prediction(folder, scenario_words):
    files = list(Path(folder).rglob("*.asc")) + list(Path(folder).rglob("*.tif"))

    files = [
        f for f in files
        if "clamping" not in f.name.lower()
        and "novel" not in f.name.lower()
        and "limiting" not in f.name.lower()
        and "background" not in f.name.lower()
    ]

    scenario_words = [w.lower() for w in scenario_words]

    matches = [
        f for f in files
        if any(w in f.name.lower() for w in scenario_words)
    ]

    candidates = matches if matches else files

    if not candidates:
        raise FileNotFoundError(f"No prediction raster found in {folder}")

    chosen = sorted(
        candidates,
        key=lambda p: (
            "median" not in p.name.lower(),
            "avg" not in p.name.lower(),
            p.name.lower()
        )
    )[0]

    print("Prediction chosen:", chosen)
    return chosen

def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan
        extent = plotting_extent(src)

    return arr, extent

def get_boundaries(countries):
    region = box(*STUDY_BOUNDS)

    region_gdf = gpd.GeoDataFrame(
        geometry=[region],
        crs="EPSG:4326"
    )

    clipped = countries[countries.intersects(region)].copy()

    if clipped.empty:
        return clipped

    return gpd.clip(clipped, region_gdf)

def normalize_each_map(arr):
    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        raise ValueError("No valid values found for normalization.")

    vmin = np.nanpercentile(finite, 2)
    vmax = np.nanpercentile(finite, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)
    norm[~np.isfinite(arr)] = np.nan

    return norm

def plot_future_map(arr, extent, countries, title, out_png):
    minx, miny, maxx, maxy = STUDY_BOUNDS
    boundaries = get_boundaries(countries)

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=0,
        vmax=1
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=15)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Relative habitat suitability")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

def make_two_panel(ssp126_png, ssp585_png, final_png):
    imgs = [
        ("SSP126 Low Carbon Emission", plt.imread(ssp126_png)),
        ("SSP585 High Carbon Emission", plt.imread(ssp585_png)),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (label, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(label, fontsize=15)

    fig.suptitle(SPECIES, fontsize=20)

    plt.tight_layout()
    plt.savefig(final_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    ensure_exists(SELECTED_CSV, "Future selected variables CSV")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LOW_FUTURE_TIF, "SSP126 future raster")
    ensure_exists(HIGH_FUTURE_TIF, "SSP585 future raster")

    bio_ids = load_selected_bio_ids()
    points = read_occurrences()

    if len(points) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(points)
    region = build_study_polygon()
    countries = load_countries()

    current_crop = OUT_DIR / "current_crop"
    ssp126_crop = OUT_DIR / "ssp126_crop"
    ssp585_crop = OUT_DIR / "ssp585_crop"

    ascii_current = OUT_DIR / "ascii_current"
    ascii_ssp126 = OUT_DIR / "ascii_ssp126"
    ascii_ssp585 = OUT_DIR / "ascii_ssp585"

    for folder in [
        current_crop,
        ssp126_crop,
        ssp585_crop,
        ascii_current,
        ascii_ssp126,
        ascii_ssp585
    ]:
        remove_and_recreate_dir(folder)

    current_layers = find_current_bio_layers(CURRENT_DIR, bio_ids)

    for bio_id, src in current_layers.items():
        crop_current_raster(
            src,
            current_crop / f"bio_{bio_id}.tif",
            region
        )

    for bio_id in bio_ids:
        extract_future_band(
            LOW_FUTURE_TIF,
            bio_id,
            region,
            ssp126_crop / f"bio_{bio_id}.tif"
        )

        extract_future_band(
            HIGH_FUTURE_TIF,
            bio_id,
            region,
            ssp585_crop / f"bio_{bio_id}.tif"
        )

    for tif in current_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_current / tif.with_suffix(".asc").name
        )

    for tif in ssp126_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_ssp126 / tif.with_suffix(".asc").name
        )

    for tif in ssp585_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_ssp585 / tif.with_suffix(".asc").name
        )

    print("ASCII current:", len(list(ascii_current.glob("*.asc"))))
    print("ASCII SSP126:", len(list(ascii_ssp126.glob("*.asc"))))
    print("ASCII SSP585:", len(list(ascii_ssp585.glob("*.asc"))))

    maxent_current = OUT_DIR / "maxent_current"
    maxent_ssp126 = OUT_DIR / "maxent_future_ssp126"
    maxent_ssp585 = OUT_DIR / "maxent_future_ssp585"

    run_maxent(
        samples_csv,
        ascii_current,
        ascii_current,
        maxent_current
    )

    run_maxent(
        samples_csv,
        ascii_current,
        ascii_ssp126,
        maxent_ssp126
    )

    run_maxent(
        samples_csv,
        ascii_current,
        ascii_ssp585,
        maxent_ssp585
    )

    pred126 = find_prediction(
        maxent_ssp126,
        ["ssp126", "future_low", "low"]
    )

    pred585 = find_prediction(
        maxent_ssp585,
        ["ssp585", "future_high", "high"]
    )

    arr126, extent126 = load_raster(pred126)
    arr585, extent585 = load_raster(pred585)

    arr126_norm = normalize_each_map(arr126)
    arr585_norm = normalize_each_map(arr585)

    plot_future_map(
        arr126_norm,
        extent126,
        countries,
        "SSP126 Low Carbon Emission",
        SSP126_PNG
    )

    plot_future_map(
        arr585_norm,
        extent585,
        countries,
        "SSP585 High Carbon Emission",
        SSP585_PNG
    )

    make_two_panel(
        SSP126_PNG,
        SSP585_PNG,
        FINAL_2PANEL
    )

    print("\nDONE ✅")
    print("Final 2-panel saved here:")
    print(FINAL_2PANEL)

if __name__ == "__main__":
    main()

Using future selected BIO layers: [1, 4, 8, 12, 14, 15]
Clean occurrence points: 56
ASCII current: 6
ASCII SSP126: 6
ASCII SSP585: 6

Running MaxEnt:
C:\Program Files\Java\jdk-17\bin\java.exe -Xmx4g -jar C:\Users\ual-laptop\Desktop\CAPSTONE\maxent\maxent.jar samplesfile=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future\Trachypithecus_phayrei_maxent_samples.csv environmentallayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future\ascii_current projectionlayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future\ascii_current outputdirectory=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future\maxent_current outputformat=cloglog randomtestpoints=25 maximumbackground=10000 maximumiterations=5000 autorun redoifexists novisible pictures=false responsecurves=false jackknife=false writebackgroundpredictions=false



Runn

In [1]:
# ============================================================
# FULL CORRECTED FUTURE SDM PIPELINE FOR Trachypithecus phayrei
# SPECIE_01_PD0602
#
# IMPORTANT FIX:
# Uses SAME BIO layers as past/current prediction:
# 02_results/correlation/Trachypithecus_phayrei_selected_variables.csv
#
# Produces:
# 1) SSP126 future prediction
# 2) SSP585 future prediction
# 3) Clear 2-panel future map
# ============================================================

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
from rasterio.features import geometry_mask
from shapely.geometry import box
import matplotlib.pyplot as plt


# ============================================================
# SETTINGS
# ============================================================

SPECIES = "Trachypithecus phayrei"
SPECIES_SAFE = SPECIES.replace(" ", "_")

BASE_DIR = Path(r"C:\Users\ual-laptop\Desktop\CAPSTONE")
PROJECT_DIR = BASE_DIR / "SPECIE_01_PD0602_Trachypithecus_phayrei"

OUT_DIR = PROJECT_DIR / "02_results" / "future_same_bio_layers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: SAME SELECTED BIO LAYERS AS PAST/CURRENT
CORR_DIR = PROJECT_DIR / "02_results" / "correlation"
SELECTED_CSV = CORR_DIR / f"{SPECIES_SAFE}_selected_variables.csv"

JAVA_EXE = Path(r"C:\Program Files\Java\jdk-17\bin\java.exe")
MAXENT_JAR = BASE_DIR / "maxent" / "maxent.jar"

GBIF_CSV = BASE_DIR / "data" / "gbif" / "Trachypithecus phayrei.csv"

COUNTRIES_SHP = (
    BASE_DIR
    / "data"
    / "boundaries"
    / "ne_10m_admin_0_countries"
    / "ne_10m_admin_0_countries.shp"
)

CURRENT_DIR = BASE_DIR / "Time Period" / "Current" / "10min"

LOW_FUTURE_TIF = (
    BASE_DIR
    / "Time Period"
    / "Future_CMIP6"
    / "MIROC6"
    / "wc2.1_2.5m_bioc_MIROC6_ssp126_2081-2100.tif"
)

HIGH_FUTURE_TIF = (
    BASE_DIR
    / "Time Period"
    / "Future_CMIP6"
    / "MIROC6"
    / "wc2.1_2.5m_bioc_MIROC6_ssp585_2081-2100.tif"
)

STUDY_BOUNDS = (67.12, -12.6, 124.12, 39.35)

MAXENT_REPLICATES = 10
MAXENT_RANDOM_TEST = 25
MAXENT_BACKGROUND = 10000
MAXENT_ITERATIONS = 5000
MAXENT_OUTPUT_FORMAT = "cloglog"
REG_MULTIPLIER = 1.0

NODATA = -9999.0
FIG_DPI = 300
CMAP = "RdYlGn_r"


# ============================================================
# OUTPUT FILES
# ============================================================

SSP126_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP126_future_clear.png"
SSP585_PNG = OUT_DIR / f"{SPECIES_SAFE}_SSP585_future_clear.png"

FINAL_2PANEL = OUT_DIR / f"{SPECIES_SAFE}_Future_2panel_same_BIO_layers.png"


# ============================================================
# BASIC HELPERS
# ============================================================

def ensure_exists(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def remove_and_recreate_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def get_safe_nodata(src_nodata):
    if src_nodata is None:
        return NODATA

    try:
        val = float(src_nodata)
        if np.isfinite(val):
            return val
    except Exception:
        pass

    return NODATA


# ============================================================
# LOAD SAME SELECTED BIO VARIABLES AS PAST/CURRENT
# ============================================================

def load_selected_bio_ids():
    ensure_exists(SELECTED_CSV, "Past/current correlation selected variables CSV")

    df = pd.read_csv(SELECTED_CSV)

    if "selected_variable" in df.columns:
        col = "selected_variable"
    elif "kept_variable" in df.columns:
        col = "kept_variable"
    elif "Variable" in df.columns and "Status" in df.columns:
        df = df[df["Status"].astype(str).str.lower() == "retained"]
        col = "Variable"
    else:
        col = df.columns[0]

    bio_ids = []

    for v in df[col].dropna():
        v = str(v).strip().lower()
        v = v.replace("bio_", "").replace("bio", "")
        bio_ids.append(int(v))

    if not bio_ids:
        raise ValueError("No selected BIO variables found in selected variables file.")

    print("Using SAME BIO layers as past/current prediction:", bio_ids)
    return bio_ids


# ============================================================
# OCCURRENCE DATA
# ============================================================

def read_occurrences():
    ensure_exists(GBIF_CSV, "GBIF CSV")

    df = pd.read_csv(GBIF_CSV)

    lon_col = next(
        (c for c in ["decimalLongitude", "longitude", "lon", "x"] if c in df.columns),
        None
    )

    lat_col = next(
        (c for c in ["decimalLatitude", "latitude", "lat", "y"] if c in df.columns),
        None
    )

    if lon_col is None or lat_col is None:
        raise ValueError(f"Longitude/latitude columns not found. Columns: {list(df.columns)}")

    df = df[[lon_col, lat_col]].dropna().copy()
    df.columns = ["lon", "lat"]

    df = df[
        (df["lon"] >= -180) & (df["lon"] <= 180) &
        (df["lat"] >= -90) & (df["lat"] <= 90)
    ].copy()

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    minx, miny, maxx, maxy = STUDY_BOUNDS

    gdf = gdf[
        (gdf.geometry.x >= minx) &
        (gdf.geometry.x <= maxx) &
        (gdf.geometry.y >= miny) &
        (gdf.geometry.y <= maxy)
    ].copy()

    gdf["x_round"] = gdf.geometry.x.round(6)
    gdf["y_round"] = gdf.geometry.y.round(6)

    gdf = gdf.drop_duplicates(subset=["x_round", "y_round"]).copy()
    gdf = gdf.drop(columns=["x_round", "y_round"])

    print("Clean occurrence points:", len(gdf))

    return gdf


def export_samples(points_gdf):
    samples_csv = OUT_DIR / f"{SPECIES_SAFE}_maxent_samples.csv"

    pd.DataFrame({
        "species": SPECIES,
        "longitude": points_gdf.geometry.x,
        "latitude": points_gdf.geometry.y
    }).to_csv(samples_csv, index=False)

    return samples_csv


# ============================================================
# SPATIAL HELPERS
# ============================================================

def build_study_polygon():
    return gpd.GeoDataFrame(
        {"name": ["study_region"]},
        geometry=[box(*STUDY_BOUNDS)],
        crs="EPSG:4326"
    )


def load_countries():
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")

    countries = gpd.read_file(COUNTRIES_SHP)

    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")

    return countries.to_crs("EPSG:4326")


# ============================================================
# RASTER PREPARATION
# ============================================================

def find_current_bio_layers(folder, bio_ids):
    folder = Path(folder)
    tif_files = list(folder.glob("*.tif"))

    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in {folder}")

    mapping = {}

    for bio_id in bio_ids:
        candidates = []

        for p in tif_files:
            stem = p.stem.lower().replace("-", "_")

            if (
                stem == f"bio_{bio_id}"
                or stem == f"bio{bio_id}"
                or stem == f"bio{bio_id:02d}"
                or f"bio_{bio_id}" in stem
                or f"bio{bio_id:02d}" in stem
                or f"bio{bio_id}" in stem
            ):
                candidates.append(p)

        if not candidates:
            raise FileNotFoundError(f"Could not find BIO{bio_id} in {folder}")

        mapping[bio_id] = sorted(candidates)[0]

    return mapping


def crop_single_raster(src_path, out_path, region):
    with rasterio.open(src_path) as src:
        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            filled=False
        )

        nodata = get_safe_nodata(src.nodata)

        arr = np.ma.array(img[0], copy=False).astype("float32")
        arr = arr.filled(nodata).astype("float32")
        arr = np.where(np.isfinite(arr), arr, nodata).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": nodata
        })

        out_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)


def extract_future_band(src_tif, bio_id, region, out_tif):
    with rasterio.open(src_tif) as src:
        if bio_id < 1 or bio_id > src.count:
            raise ValueError(
                f"BIO{bio_id} requested, but future raster has only {src.count} bands."
            )

        img, transform = mask(
            src,
            list(region.geometry),
            crop=True,
            indexes=bio_id,
            filled=False
        )

        nodata = get_safe_nodata(src.nodata)

        arr = np.ma.array(img, copy=False).astype("float32")
        arr = arr.filled(nodata).astype("float32")
        arr = np.where(np.isfinite(arr), arr, nodata).astype("float32")

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "count": 1,
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": transform,
            "dtype": "float32",
            "nodata": nodata
        })

        out_tif.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(arr, 1)


def tif_to_ascii(src_tif, out_asc):
    with rasterio.open(src_tif) as src:
        band = src.read(1, masked=True).astype("float32")

        nodata = get_safe_nodata(src.nodata)

        arr = np.ma.array(band, copy=False).filled(nodata).astype("float32")
        arr = np.where(np.isfinite(arr), arr, nodata).astype("float32")

        transform = src.transform
        nrows, ncols = arr.shape

        xll = transform.c
        yll = transform.f + transform.e * nrows
        cellsize = transform.a

        out_asc.parent.mkdir(parents=True, exist_ok=True)

        with open(out_asc, "w") as f:
            f.write(f"ncols        {ncols}\n")
            f.write(f"nrows        {nrows}\n")
            f.write(f"xllcorner    {xll}\n")
            f.write(f"yllcorner    {yll}\n")
            f.write(f"cellsize     {cellsize}\n")
            f.write(f"NODATA_value {nodata}\n")

            for row in arr:
                f.write(" ".join(f"{float(v):.6f}" for v in row) + "\n")


def validate_ascii_file(path):
    with open(path, "r") as f:
        lines = f.readlines()

    if len(lines) < 7:
        raise ValueError(f"ASCII file too short: {path}")

    header = lines[:6]
    data = lines[6:]

    ncols = int(header[0].split()[1])
    nrows = int(header[1].split()[1])

    if len(data) != nrows:
        raise ValueError(f"{path.name}: expected {nrows} rows, found {len(data)}")

    for i, row in enumerate(data, start=1):
        values = row.strip().split()

        if len(values) != ncols:
            raise ValueError(
                f"{path.name}: row {i} expected {ncols} columns, found {len(values)}"
            )

        if any(v.lower() == "nan" for v in values):
            raise ValueError(f"{path.name}: contains literal nan on row {i}")


def validate_ascii_directory(folder):
    folder = Path(folder)
    asc_files = list(folder.glob("*.asc"))

    if not asc_files:
        raise ValueError(f"No ASCII files found in {folder}")

    for asc in asc_files:
        validate_ascii_file(asc)


# ============================================================
# MAXENT
# ============================================================

def run_maxent(samples_csv, env_dir, projection_dir, out_dir):
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(samples_csv, "samples CSV")
    ensure_exists(env_dir, "environment directory")
    ensure_exists(projection_dir, "projection directory")

    validate_ascii_directory(env_dir)
    validate_ascii_directory(projection_dir)

    remove_and_recreate_dir(out_dir)

    cmd = [
        str(JAVA_EXE),
        "-Xmx4g",
        "-jar",
        str(MAXENT_JAR),
        f"samplesfile={samples_csv}",
        f"environmentallayers={env_dir}",
        f"projectionlayers={projection_dir}",
        f"outputdirectory={out_dir}",
        f"randomtestpoints={MAXENT_RANDOM_TEST}",
        f"maximumbackground={MAXENT_BACKGROUND}",
        f"replicates={MAXENT_REPLICATES}",
        "replicatetype=subsample",
        f"maximumiterations={MAXENT_ITERATIONS}",
        f"outputformat={MAXENT_OUTPUT_FORMAT}",
        f"betamultiplier={REG_MULTIPLIER}",
        "autofeature=true",
        "responsecurves=true",
        "jackknife=true",
        "pictures=true",
        "writebackgroundpredictions=false",
        "visible=false",
        "autorun",
        "redoifexists",
        "novisible"
    ]

    print("\n============================================================")
    print("Running MaxEnt")
    print("============================================================")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- MaxEnt STDOUT ---")
    print(result.stdout)

    print("\n--- MaxEnt STDERR ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("MaxEnt failed. Check STDERR output above.")


def find_prediction(output_dir):
    output_dir = Path(output_dir)

    candidates = list(output_dir.glob("*.tif")) + list(output_dir.glob("*.asc"))

    candidates = [
        p for p in candidates
        if "clamping" not in p.name.lower()
        and "novel" not in p.name.lower()
        and "limiting" not in p.name.lower()
        and "background" not in p.name.lower()
    ]

    if not candidates:
        raise FileNotFoundError(f"No prediction raster found in {output_dir}")

    ranked = sorted(
        candidates,
        key=lambda p: (
            "median" not in p.stem.lower(),
            "avg" not in p.stem.lower(),
            p.name.lower()
        )
    )

    chosen = ranked[0]
    print("Prediction chosen:", chosen)

    return chosen


# ============================================================
# PLOTTING
# ============================================================

def load_land_masked_prediction(raster_path, countries):
    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype(float)

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        arr[np.isinf(arr)] = np.nan

        extent = plotting_extent(src)

        xmin, xmax, ymin, ymax = extent
        raster_box = box(xmin, ymin, xmax, ymax)

        countries_here = countries[countries.intersects(raster_box)].copy()

        if not countries_here.empty:
            raster_gdf = gpd.GeoDataFrame(
                {"name": ["raster_extent"]},
                geometry=[raster_box],
                crs="EPSG:4326"
            )

            countries_here = gpd.clip(countries_here, raster_gdf)

            land_mask = geometry_mask(
                countries_here.geometry,
                transform=src.transform,
                invert=True,
                out_shape=arr.shape
            )

            arr[~land_mask] = np.nan

    return arr, extent, countries_here


def normalize_for_clear_display(arr):
    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        raise ValueError("No valid raster values found for normalization.")

    vmin = np.nanpercentile(finite, 2)
    vmax = np.nanpercentile(finite, 98)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(finite)
        vmax = np.nanmax(finite)

    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)
    norm[~np.isfinite(arr)] = np.nan

    return norm


def plot_future_map(raster_path, countries, title, out_png):
    arr, extent, boundaries = load_land_masked_prediction(raster_path, countries)

    arr_norm = normalize_for_clear_display(arr)

    minx, miny, maxx, maxy = STUDY_BOUNDS

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(
        arr_norm,
        extent=extent,
        origin="upper",
        cmap=CMAP,
        vmin=0,
        vmax=1
    )

    if not boundaries.empty:
        boundaries.boundary.plot(ax=ax, color="black", linewidth=0.5)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    cbar.set_label("Relative habitat suitability")

    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()


def make_two_panel(ssp126_png, ssp585_png, final_png):
    imgs = [
        ("SSP126 Low Carbon Emission", plt.imread(ssp126_png)),
        ("SSP585 High Carbon Emission", plt.imread(ssp585_png)),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (label, img) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(label, fontsize=15)

    fig.suptitle(f"{SPECIES} - Future SDM Projection", fontsize=20)

    plt.tight_layout()
    plt.savefig(final_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN
# ============================================================

def main():
    print("Output folder:", OUT_DIR)

    ensure_exists(SELECTED_CSV, "Past/current selected variables CSV")
    ensure_exists(JAVA_EXE, "java.exe")
    ensure_exists(MAXENT_JAR, "maxent.jar")
    ensure_exists(GBIF_CSV, "GBIF CSV")
    ensure_exists(COUNTRIES_SHP, "Countries shapefile")
    ensure_exists(CURRENT_DIR, "Current climate folder")
    ensure_exists(LOW_FUTURE_TIF, "SSP126 future raster")
    ensure_exists(HIGH_FUTURE_TIF, "SSP585 future raster")

    selected_bio_ids = load_selected_bio_ids()

    points = read_occurrences()

    if len(points) < 10:
        raise ValueError("Too few occurrence points after cleaning.")

    samples_csv = export_samples(points)

    study_region = build_study_polygon()
    countries = load_countries()

    current_crop = OUT_DIR / "current_crop"
    ssp126_crop = OUT_DIR / "ssp126_crop"
    ssp585_crop = OUT_DIR / "ssp585_crop"

    ascii_current = OUT_DIR / "ascii_current"
    ascii_ssp126 = OUT_DIR / "ascii_ssp126"
    ascii_ssp585 = OUT_DIR / "ascii_ssp585"

    for folder in [
        current_crop,
        ssp126_crop,
        ssp585_crop,
        ascii_current,
        ascii_ssp126,
        ascii_ssp585
    ]:
        remove_and_recreate_dir(folder)

    print("\nFinding current BIO layers...")
    current_layers = find_current_bio_layers(CURRENT_DIR, selected_bio_ids)

    print("\nCropping current BIO layers...")
    for bio_id, src in current_layers.items():
        crop_single_raster(
            src,
            current_crop / f"bio_{bio_id}.tif",
            study_region
        )

    print("\nExtracting same BIO bands from SSP126 and SSP585 future rasters...")
    for bio_id in selected_bio_ids:
        extract_future_band(
            LOW_FUTURE_TIF,
            bio_id,
            study_region,
            ssp126_crop / f"bio_{bio_id}.tif"
        )

        extract_future_band(
            HIGH_FUTURE_TIF,
            bio_id,
            study_region,
            ssp585_crop / f"bio_{bio_id}.tif"
        )

    print("\nConverting current and future rasters to ASCII...")

    for tif in current_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_current / tif.with_suffix(".asc").name
        )

    for tif in ssp126_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_ssp126 / tif.with_suffix(".asc").name
        )

    for tif in ssp585_crop.glob("*.tif"):
        tif_to_ascii(
            tif,
            ascii_ssp585 / tif.with_suffix(".asc").name
        )

    print("\nASCII file counts:")
    print("Current:", len(list(ascii_current.glob("*.asc"))))
    print("SSP126:", len(list(ascii_ssp126.glob("*.asc"))))
    print("SSP585:", len(list(ascii_ssp585.glob("*.asc"))))

    maxent_current = OUT_DIR / "maxent_current"
    maxent_ssp126 = OUT_DIR / "maxent_future_ssp126"
    maxent_ssp585 = OUT_DIR / "maxent_future_ssp585"

    print("\nRunning current MaxEnt model...")
    run_maxent(
        samples_csv,
        ascii_current,
        ascii_current,
        maxent_current
    )

    print("\nProjecting model to SSP126 future climate...")
    run_maxent(
        samples_csv,
        ascii_current,
        ascii_ssp126,
        maxent_ssp126
    )

    print("\nProjecting model to SSP585 future climate...")
    run_maxent(
        samples_csv,
        ascii_current,
        ascii_ssp585,
        maxent_ssp585
    )

    print("\nFinding future prediction rasters...")

    pred126 = find_prediction(maxent_ssp126)
    pred585 = find_prediction(maxent_ssp585)

    print("\nCreating future maps...")

    plot_future_map(
        pred126,
        countries,
        f"{SPECIES} - SSP126 Future SDM",
        SSP126_PNG
    )

    plot_future_map(
        pred585,
        countries,
        f"{SPECIES} - SSP585 Future SDM",
        SSP585_PNG
    )

    make_two_panel(
        SSP126_PNG,
        SSP585_PNG,
        FINAL_2PANEL
    )

    print("\n============================================================")
    print("DONE ✅")
    print("============================================================")
    print("Used SAME BIO layers as past/current prediction:")
    print(selected_bio_ids)
    print("\nFinal 2-panel future map saved here:")
    print(FINAL_2PANEL)


if __name__ == "__main__":
    main()

Output folder: C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future_same_bio_layers
Using SAME BIO layers as past/current prediction: [1, 4, 8, 10, 12, 15]
Clean occurrence points: 56

Finding current BIO layers...

Cropping current BIO layers...

Extracting same BIO bands from SSP126 and SSP585 future rasters...

Converting current and future rasters to ASCII...

ASCII file counts:
Current: 6
SSP126: 6
SSP585: 6

Running current MaxEnt model...

Running MaxEnt
C:\Program Files\Java\jdk-17\bin\java.exe -Xmx4g -jar C:\Users\ual-laptop\Desktop\CAPSTONE\maxent\maxent.jar samplesfile=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future_same_bio_layers\Trachypithecus_phayrei_maxent_samples.csv environmentallayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Trachypithecus_phayrei\02_results\future_same_bio_layers\ascii_current projectionlayers=C:\Users\ual-laptop\Desktop\CAPSTONE\SPECIE_01_PD0602_Tr